# Notebook 13 — Prompt Caching

Use `cache_control` to cache large system prompts / documents and reduce latency + cost on repeated calls.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL, MAIN_MODEL


## Without caching — baseline

In [ ]:
LONG_DOCUMENT = ("This is a long document that contains many facts. " * 100)

def ask_no_cache(question):
    t0 = time.time()
    r = client.messages.create(
        model=MAIN_MODEL, max_tokens=200,
        system=f"Answer questions using this document:\n\n{LONG_DOCUMENT}",
        messages=[{"role": "user", "content": question}],
    )
    return r.content[0].text, time.time()-t0, r.usage

reply, elapsed, usage = ask_no_cache("Summarise the document.")
print(f"Reply: {reply[:80]}...")
print(f"Time: {elapsed:.2f}s | Input tokens: {usage.input_tokens}")


## With `cache_control: ephemeral`

In [ ]:
def ask_with_cache(question):
    t0 = time.time()
    r = client.messages.create(
        model=MAIN_MODEL, max_tokens=200,
        system=[
            {
                "type": "text",
                "text": f"Answer questions using this document:\n\n{LONG_DOCUMENT}",
                "cache_control": {"type": "ephemeral"},   # cache this block
            }
        ],
        messages=[{"role": "user", "content": question}],
    )
    return r.content[0].text, time.time()-t0, r.usage

print("First call (cache miss):")
_, t1, u1 = ask_with_cache("Summarise the document.")
print(f"  Time: {t1:.2f}s | cache_read: {getattr(u1,'cache_read_input_tokens',0)} | cache_create: {getattr(u1,'cache_creation_input_tokens',0)}")

print("\nSecond call (cache hit):")
_, t2, u2 = ask_with_cache("What is the main topic?")
print(f"  Time: {t2:.2f}s | cache_read: {getattr(u2,'cache_read_input_tokens',0)} | cache_create: {getattr(u2,'cache_creation_input_tokens',0)}")


## When to use caching
- Large, repeated system prompts (docs, codebases, policy files)
- Cache TTL is 5 minutes after last use
- Cache hit saves ~90% latency on the cached portion

## Exercise
Measure the latency improvement across 5 consecutive calls. Plot or print the times.

In [ ]:
# Your code here
